# Session 05 · MongoDB Database Interaction

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - connect and insert

In [2]:
from mongo_conn import get_db, backend
print("Backend:", backend())
db = get_db("shop_demo")
db.orders.delete_many({})
db.orders.insert_many([
    {"customer": "Ali", "city": "Karachi", "items": [{"name": "kurta", "qty": 2, "price": 2500}], "status": "paid"},
    {"customer": "Sana", "city": "Lahore", "items": [{"name": "shawl", "qty": 1, "price": 4200}], "status": "pending"},
    {"customer": "Hina", "city": "Karachi", "items": [{"name": "kurta", "qty": 1, "price": 2600},
                                                       {"name": "dupatta", "qty": 3, "price": 900}], "status": "paid"},
])
print(db.orders.count_documents({}), "documents")

Backend: in-memory mongomock
3 documents


## Example 2 - filters and projections

In [3]:
for doc in db.orders.find({"city": "Karachi"}, {"_id": 0, "customer": 1, "status": 1}):
    print(doc)

{'customer': 'Ali', 'status': 'paid'}
{'customer': 'Hina', 'status': 'paid'}


## Example 3 - nested fields and arrays

In [4]:
print([d["customer"] for d in db.orders.find({"items.name": "kurta"})])
print([d["customer"] for d in db.orders.find({"items.qty": {"$gte": 2}})])

['Ali', 'Hina']
['Ali', 'Hina']


## Example 4 - update and count

In [5]:
res = db.orders.update_many({"status": "pending"}, {"$set": {"status": "paid"}})
print("modified:", res.modified_count, "| paid:", db.orders.count_documents({"status": "paid"}))

modified: 1 | paid: 3


## Example 5 - aggregation: revenue per city

In [6]:
pipeline = [
    {"$unwind": "$items"},
    {"$group": {"_id": "$city", "revenue": {"$sum": {"$multiply": ["$items.qty", "$items.price"]}}}},
    {"$sort": {"revenue": -1}},
]
result = list(db.orders.aggregate(pipeline))
print(result)
assert result[0] == {"_id": "Karachi", "revenue": 10300}

[{'revenue': 10300, '_id': 'Karachi'}, {'revenue': 4200, '_id': 'Lahore'}]


## Example 6 - into pandas

In [7]:
import pandas as pd
df = pd.json_normalize(list(db.orders.find({}, {"_id": 0})), record_path="items",
                       meta=["customer", "city", "status"])
df["line_total"] = df["qty"] * df["price"]
print(df)
print(df.groupby("name")["line_total"].sum())

      name  qty  price customer     city status  line_total
0    kurta    2   2500      Ali  Karachi   paid        5000
1    shawl    1   4200     Sana   Lahore   paid        4200
2    kurta    1   2600     Hina  Karachi   paid        2600
3  dupatta    3    900     Hina  Karachi   paid        2700
name
dupatta    2700
kurta      7600
shawl      4200
Name: line_total, dtype: int64
